# Handcrafted Features + CatBoost

This notebook extends the reproduced Kaggle baseline with a boosting model
and additional handcrafted features.

Experiments:
1. Logistic Regression + original 9 features
2. CatBoost + original 9 features
3. Logistic Regression + extended features
4. CatBoost + extended features
5. Feature-group ablation

All experiments use the same 5-fold stratified cross-validation splits
(random_state=42) for a fair comparison.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostClassifier

In [2]:
DATA_DIR = Path("../data/raw")

tr = pd.read_csv(DATA_DIR / "train.csv")
te = pd.read_csv(DATA_DIR / "test.csv")


def parse_text(s):
    try:
        return " ".join(x or "" for x in json.loads(s))
    except Exception:
        return str(s)


for col in ["prompt", "response_a", "response_b"]:
    tr[col + "_txt"] = tr[col].apply(parse_text)
    te[col + "_txt"] = te[col].apply(parse_text)


y = np.argmax(
    tr[["winner_model_a", "winner_model_b", "winner_tie"]].values,
    axis=1
)

print("Train shape:", tr.shape)
print("Classes:", np.bincount(y))

Train shape: (57477, 12)
Classes: [20064 19652 17761]


In [3]:
MD_PAT = r'\n[*#-]|\n\d+\.|\*\*'

REFUSAL = re.compile(
    r"I cannot|I can't|I'm sorry|As an AI|I am unable|I apologize",
    re.I
)

MD = re.compile(MD_PAT)


FEATURE_NAMES_9 = [
    "len_ratio_a_b",
    "log_len_a",
    "log_len_b",
    "markdown_a",
    "markdown_b",
    "refusals_a",
    "refusals_b",
    "similar_length",
    "prompt_length",
]


def original_features(df):
    la = df.response_a_txt.str.len().to_numpy(float)
    lb = df.response_b_txt.str.len().to_numpy(float)

    log_diff = np.log1p(la) - np.log1p(lb)

    return np.column_stack([
        log_diff,
        np.log1p(la),
        np.log1p(lb),

        df.response_a_txt.str.count(MD).to_numpy(float),
        df.response_b_txt.str.count(MD).to_numpy(float),

        df.response_a_txt.str.count(REFUSAL).to_numpy(float),
        df.response_b_txt.str.count(REFUSAL).to_numpy(float),

        (np.abs(log_diff) < 0.15).astype(float),

        df.prompt_txt.str.len().to_numpy(float) / 1000.0,
    ])


X9 = original_features(tr)

print("Feature matrix:", X9.shape)

pd.DataFrame(X9, columns=FEATURE_NAMES_9).head()

Feature matrix: (57477, 9)


,len_ratio_a_b,log_len_a,log_len_b,markdown_a,markdown_b,refusals_a,refusals_b,similar_length,prompt_length
0,1.311994,8.402904,7.090910,32.0,0.0,0.0,2.0,0.0,0.159
1,-0.148554,8.038189,8.186743,3.0,5.0,0.0,0.0,1.0,0.192
2,-0.704942,6.785588,7.490529,2.0,0.0,0.0,0.0,0.0,0.056
3,0.713569,8.060224,7.346655,10.0,5.0,0.0,0.0,0.0,0.083
4,0.528539,7.163172,6.634633,3.0,0.0,0.0,0.0,0.0,0.075


In [4]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

CV_SPLITS = list(skf.split(X9, y))

print("Number of folds:", len(CV_SPLITS))

Number of folds: 5


In [5]:
def evaluate_logistic_regression(X, y, splits):
    oof = np.zeros((len(y), 3))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(splits):
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X[train_idx])
        X_val = scaler.transform(X[val_idx])

        model = LogisticRegression(
            max_iter=2000
        )

        model.fit(X_train, y[train_idx])

        pred = model.predict_proba(X_val)
        oof[val_idx] = pred

        score = log_loss(y[val_idx], pred)
        fold_scores.append(score)

        print(f"Fold {fold}: {score:.4f}")

    oof_score = log_loss(y, oof)

    print(f"\nOOF log loss: {oof_score:.4f}")
    print(f"Fold mean:     {np.mean(fold_scores):.4f}")
    print(f"Fold std:      {np.std(fold_scores):.4f}")

    return {
        "oof_predictions": oof,
        "fold_scores": fold_scores,
        "log_loss": oof_score,
    }


lr_9_result = evaluate_logistic_regression(
    X9,
    y,
    CV_SPLITS
)

Fold 0: 1.0634
Fold 1: 1.0633
Fold 2: 1.0596
Fold 3: 1.0613
Fold 4: 1.0659

OOF log loss: 1.0627
Fold mean:     1.0627
Fold std:      0.0021


In [6]:
def evaluate_catboost(X, y, splits):
    oof = np.zeros((len(y), 3))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(splits):
        model = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=500,
            learning_rate=0.05,
            depth=6,
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
        )

        model.fit(
            X[train_idx],
            y[train_idx],
        )

        pred = model.predict_proba(X[val_idx])
        oof[val_idx] = pred

        score = log_loss(y[val_idx], pred)
        fold_scores.append(score)

        print(f"Fold {fold}: {score:.4f}")

    oof_score = log_loss(y, oof)

    print(f"\nOOF log loss: {oof_score:.4f}")
    print(f"Fold mean:     {np.mean(fold_scores):.4f}")
    print(f"Fold std:      {np.std(fold_scores):.4f}")

    return {
        "oof_predictions": oof,
        "fold_scores": fold_scores,
        "log_loss": oof_score,
    }


catboost_9_result = evaluate_catboost(
    X9,
    y,
    CV_SPLITS
)

Fold 0: 1.0408
Fold 1: 1.0433
Fold 2: 1.0407
Fold 3: 1.0397
Fold 4: 1.0450

OOF log loss: 1.0419
Fold mean:     1.0419
Fold std:      0.0020


In [7]:
comparison_9 = pd.DataFrame({
    "model": [
        "Logistic Regression + 9 features",
        "CatBoost + 9 features",
    ],
    "log_loss": [
        lr_9_result["log_loss"],
        catboost_9_result["log_loss"],
    ],
})

comparison_9

,model,log_loss
0,Logistic Regression + 9 features,1.062693
1,CatBoost + 9 features,1.041892


In [8]:
delta = (
    catboost_9_result["log_loss"]
    - lr_9_result["log_loss"]
)

print(f"CatBoost - LR: {delta:+.4f}")

CatBoost - LR: -0.0208


## Extended handcrafted features

The extended feature set contains all 9 original baseline features and
additional response-structure features inspired by the 21st-place LMSYS
solution.

Since the original feature extraction code was not publicly available,
the exact definitions used here are documented explicitly.

In [9]:
import string


WORD_RE = re.compile(r"\b\w+\b", re.UNICODE)


def get_words(text):
    """Extract lowercase word-like tokens."""
    if not isinstance(text, str):
        return []

    return WORD_RE.findall(text.lower())


def get_sentences(text):
    """
    Approximate sentence splitting using '.', '!' and '?'.
    Empty sentences are removed.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def get_paragraphs(text):
    """
    Paragraphs are non-empty text blocks separated by
    one or more newline characters.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    paragraphs = re.split(r"\n+", text.strip())

    return [
        paragraph.strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

In [10]:
def response_structure_features(text):
    words = get_words(text)
    sentences = get_sentences(text)
    paragraphs = get_paragraphs(text)

    # ---------- Length ----------
    length = len(text) if isinstance(text, str) else 0

    # ---------- Sentence statistics ----------
    sentence_lengths = [
        len(get_words(sentence))
        for sentence in sentences
    ]

    long_sents = sum(
        sent_len >= 30
        for sent_len in sentence_lengths
    )

    short_sents = sum(
        sent_len <= 10
        for sent_len in sentence_lengths
    )

    avg_sent_len = (
        np.mean(sentence_lengths)
        if sentence_lengths
        else 0.0
    )

    sd_sent_len = (
        np.std(sentence_lengths)
        if sentence_lengths
        else 0.0
    )

    # ---------- Paragraph statistics ----------
    paragraph_lengths = [
        len(get_words(paragraph))
        for paragraph in paragraphs
    ]

    num_paragraphs = len(paragraphs)

    mean_paragraph_length = (
        np.mean(paragraph_lengths)
        if paragraph_lengths
        else 0.0
    )

    sd_paragraph_length = (
        np.std(paragraph_lengths)
        if paragraph_lengths
        else 0.0
    )

    sentences_per_paragraph = [
        len(get_sentences(paragraph))
        for paragraph in paragraphs
    ]

    avg_sent_per_paragraph = (
        np.mean(sentences_per_paragraph)
        if sentences_per_paragraph
        else 0.0
    )

    sd_sent_per_paragraph = (
        np.std(sentences_per_paragraph)
        if sentences_per_paragraph
        else 0.0
    )

    # ---------- Repetition ----------
    if words:
        repetition_density = (
            1.0 - len(set(words)) / len(words)
        )
    else:
        repetition_density = 0.0

    # ---------- Punctuation ----------
    if length > 0:
        punctuation_count = sum(
            char in string.punctuation
            for char in text
        )

        punctuation_density = (
            punctuation_count / length
        )
    else:
        punctuation_density = 0.0

    return [
        length,
        long_sents,
        short_sents,
        avg_sent_len,
        sd_sent_len,
        num_paragraphs,
        mean_paragraph_length,
        sd_paragraph_length,
        avg_sent_per_paragraph,
        sd_sent_per_paragraph,
        repetition_density,
        punctuation_density,
    ]

In [11]:
STRUCTURE_FEATURE_NAMES = [
    "length",
    "long_sents",
    "short_sents",
    "avg_sent_len",
    "sd_sent_len",
    "num_paragraphs",
    "mean_paragraph_length",
    "sd_paragraph_length",
    "avg_sent_per_paragraph",
    "sd_sent_per_paragraph",
    "repetition_density",
    "punctuation_density",
]


EXTENDED_FEATURE_NAMES = (
    FEATURE_NAMES_9
    + [f"{name}_a" for name in STRUCTURE_FEATURE_NAMES]
    + [f"{name}_b" for name in STRUCTURE_FEATURE_NAMES]
)

print("Number of candidate features:", len(EXTENDED_FEATURE_NAMES))

Number of candidate features: 33


In [13]:
def extended_features(df):
    # Original baseline features
    X_original = original_features(df)

    # 12 structural features for response A
    features_a = np.array([
        response_structure_features(text)
        for text in df["response_a_txt"]
    ])

    # 12 structural features for response B
    features_b = np.array([
        response_structure_features(text)
        for text in df["response_b_txt"]
    ])

    # 9 + 12 + 12 = 33
    return np.column_stack([
        X_original,
        features_a,
        features_b,
    ])


X_ext = extended_features(tr)

print("Extended feature matrix:", X_ext.shape)

Extended feature matrix: (57477, 33)


In [14]:
X_ext_df = pd.DataFrame(
    X_ext,
    columns=EXTENDED_FEATURE_NAMES
)

X_ext_df.head()

,len_ratio_a_b,log_len_a,log_len_b,markdown_a,markdown_b,refusals_a,refusals_b,similar_length,prompt_length,length_a,...,short_sents_b,avg_sent_len_b,sd_sent_len_b,num_paragraphs_b,mean_paragraph_length_b,sd_paragraph_length_b,avg_sent_per_paragraph_b,sd_sent_per_paragraph_b,repetition_density_b,punctuation_density_b
0,1.311994,8.402904,7.090910,32.0,0.0,0.0,2.0,0.0,0.159,4459.0,...,1.0,17.666667,4.921608,1.0,212.000000,0.000000,12.000000,0.000000,0.396226,0.028333
1,-0.148554,8.038189,8.186743,3.0,5.0,0.0,0.0,1.0,0.192,3096.0,...,5.0,17.636364,10.224208,19.0,30.631579,30.222352,1.947368,1.468034,0.620275,0.027561
2,-0.704942,6.785588,7.490529,2.0,0.0,0.0,0.0,0.0,0.056,884.0,...,0.0,22.153846,11.614620,20.0,14.400000,22.907204,1.500000,0.974679,0.659722,0.076536
3,0.713569,8.060224,7.346655,10.0,5.0,0.0,0.0,0.0,0.083,3165.0,...,4.0,15.166667,10.329300,7.0,39.000000,6.369571,2.714286,0.880631,0.608059,0.020645
4,0.528539,7.163172,6.634633,3.0,0.0,0.0,0.0,0.0,0.075,1290.0,...,1.0,16.125000,4.910130,5.0,25.800000,8.328265,1.600000,0.800000,0.418605,0.025000


In [15]:
print("NaN values:", X_ext_df.isna().sum().sum())
print("Infinite values:", np.isinf(X_ext).sum())

X_ext_df.describe().T

NaN values: 0
Infinite values: 0


,count,mean,std,min,25%,50%,75%,max
len_ratio_a_b,57477.0,-0.005883,1.060294,-7.868637,-0.470004,0.000000,0.461346,8.481566
log_len_a,57477.0,6.616610,1.301763,0.000000,5.971262,6.944087,7.501634,10.883692
log_len_b,57477.0,6.622493,1.298648,0.000000,5.981414,6.951772,7.506592,10.866719
markdown_a,57477.0,4.321555,9.884741,0.000000,0.000000,0.000000,5.000000,270.000000
markdown_b,57477.0,4.363659,10.204925,0.000000,0.000000,0.000000,5.000000,514.000000
refusals_a,57477.0,0.132975,0.503151,0.000000,0.000000,0.000000,0.000000,14.000000
refusals_b,57477.0,0.134036,0.515461,0.000000,0.000000,0.000000,0.000000,24.000000
similar_length,57477.0,0.199141,0.399357,0.000000,0.000000,0.000000,0.000000,1.000000
prompt_length,57477.0,0.352357,1.025378,0.003000,0.048000,0.091000,0.233000,32.833000
length_a,57477.0,1330.371488,1461.908578,0.000000,391.000000,1036.000000,1810.000000,53299.000000


In [16]:
lr_ext_result = evaluate_logistic_regression(
    X_ext,
    y,
    CV_SPLITS
)

Fold 0: 1.0527
Fold 1: 1.0535
Fold 2: 1.0473
Fold 3: 1.0506
Fold 4: 1.0547

OOF log loss: 1.0518
Fold mean:     1.0518
Fold std:      0.0026


In [17]:
catboost_ext_result = evaluate_catboost(
    X_ext,
    y,
    CV_SPLITS
)

Fold 0: 1.0320
Fold 1: 1.0316
Fold 2: 1.0274
Fold 3: 1.0259
Fold 4: 1.0307

OOF log loss: 1.0295
Fold mean:     1.0295
Fold std:      0.0024


In [18]:
comparison = pd.DataFrame({
    "model": [
        "Logistic Regression + 9 features",
        "CatBoost + 9 features",
        "Logistic Regression + extended features",
        "CatBoost + extended features",
    ],
    "log_loss": [
        lr_9_result["log_loss"],
        catboost_9_result["log_loss"],
        lr_ext_result["log_loss"],
        catboost_ext_result["log_loss"],
    ],
})

comparison = comparison.sort_values("log_loss")

comparison

,model,log_loss
3,CatBoost + extended features,1.029526
1,CatBoost + 9 features,1.041892
2,Logistic Regression + extended features,1.051759
0,Logistic Regression + 9 features,1.062693


In [19]:
lr_feature_gain = (
    lr_ext_result["log_loss"]
    - lr_9_result["log_loss"]
)

catboost_feature_gain = (
    catboost_ext_result["log_loss"]
    - catboost_9_result["log_loss"]
)

print(
    f"Extended features effect on LR: "
    f"{lr_feature_gain:+.4f}"
)

print(
    f"Extended features effect on CatBoost: "
    f"{catboost_feature_gain:+.4f}"
)

Extended features effect on LR: -0.0109
Extended features effect on CatBoost: -0.0124


In [20]:
FEATURE_GROUPS = {
    "response_length": [
        "len_ratio_a_b",
        "log_len_a",
        "log_len_b",
        "similar_length",
        "length_a",
        "length_b",
    ],

    "prompt_length": [
        "prompt_length",
    ],

    "markdown": [
        "markdown_a",
        "markdown_b",
    ],

    "refusals": [
        "refusals_a",
        "refusals_b",
    ],

    "sentence_stats": [
        "long_sents_a",
        "long_sents_b",
        "short_sents_a",
        "short_sents_b",
        "avg_sent_len_a",
        "avg_sent_len_b",
        "sd_sent_len_a",
        "sd_sent_len_b",
    ],

    "paragraph_stats": [
        "num_paragraphs_a",
        "num_paragraphs_b",
        "mean_paragraph_length_a",
        "mean_paragraph_length_b",
        "sd_paragraph_length_a",
        "sd_paragraph_length_b",
        "avg_sent_per_paragraph_a",
        "avg_sent_per_paragraph_b",
        "sd_sent_per_paragraph_a",
        "sd_sent_per_paragraph_b",
    ],

    "repetition": [
        "repetition_density_a",
        "repetition_density_b",
    ],

    "punctuation": [
        "punctuation_density_a",
        "punctuation_density_b",
    ],
}

In [21]:
feature_to_idx = {
    name: i
    for i, name in enumerate(EXTENDED_FEATURE_NAMES)
}


def select_features(X, feature_names):
    indices = [
        feature_to_idx[name]
        for name in feature_names
    ]

    return X[:, indices]

In [23]:
full_catboost_loss = catboost_ext_result["log_loss"]

ablation_results = []

for group_name, group_features in FEATURE_GROUPS.items():

    kept_features = [
        feature
        for feature in EXTENDED_FEATURE_NAMES
        if feature not in group_features
    ]

    X_without_group = select_features(
        X_ext,
        kept_features
    )

    print()
    print("=" * 60)
    print(f"Removing group: {group_name}")
    print(
        f"Features: {len(EXTENDED_FEATURE_NAMES)}"
        f" -> {len(kept_features)}"
    )
    print("=" * 60)

    result = evaluate_catboost(
        X_without_group,
        y,
        CV_SPLITS
    )

    loss_without = result["log_loss"]

    delta = loss_without - full_catboost_loss

    ablation_results.append({
        "removed_group": group_name,
        "log_loss_without": loss_without,
        "delta_vs_full": delta,
        "n_removed": len(group_features),
    })


Removing group: response_length
Features: 33 -> 27
Fold 0: 1.0412
Fold 1: 1.0384
Fold 2: 1.0367
Fold 3: 1.0348
Fold 4: 1.0392

OOF log loss: 1.0380
Fold mean:     1.0380
Fold std:      0.0022

Removing group: prompt_length
Features: 33 -> 32
Fold 0: 1.0342
Fold 1: 1.0342
Fold 2: 1.0299
Fold 3: 1.0274
Fold 4: 1.0319

OOF log loss: 1.0315
Fold mean:     1.0315
Fold std:      0.0026

Removing group: markdown
Features: 33 -> 31
Fold 0: 1.0323
Fold 1: 1.0315
Fold 2: 1.0276
Fold 3: 1.0258
Fold 4: 1.0303

OOF log loss: 1.0295
Fold mean:     1.0295
Fold std:      0.0024

Removing group: refusals
Features: 33 -> 31
Fold 0: 1.0419
Fold 1: 1.0392
Fold 2: 1.0357
Fold 3: 1.0334
Fold 4: 1.0352

OOF log loss: 1.0371
Fold mean:     1.0371
Fold std:      0.0030

Removing group: sentence_stats
Features: 33 -> 25
Fold 0: 1.0318
Fold 1: 1.0307
Fold 2: 1.0273
Fold 3: 1.0255
Fold 4: 1.0311

OOF log loss: 1.0293
Fold mean:     1.0293
Fold std:      0.0024

Removing group: paragraph_stats
Features: 33 -> 23


In [24]:
ablation_df = pd.DataFrame(ablation_results)

ablation_df = ablation_df.sort_values(
    "delta_vs_full",
    ascending=False
)

ablation_df

,removed_group,log_loss_without,delta_vs_full,n_removed
0,response_length,1.038050,0.008524,6
3,refusals,1.037089,0.007563,2
6,repetition,1.034821,0.005295,2
1,prompt_length,1.031523,0.001997,1
7,punctuation,1.031068,0.001542,2
5,paragraph_stats,1.030286,0.000760,10
2,markdown,1.029510,-0.000016,2
4,sentence_stats,1.029306,-0.000219,8


In [25]:
DROP_GROUPS = [
    "markdown",
    "sentence_stats",
]

features_to_drop = set()

for group in DROP_GROUPS:
    features_to_drop.update(FEATURE_GROUPS[group])

PRUNED_FEATURE_NAMES = [
    feature
    for feature in EXTENDED_FEATURE_NAMES
    if feature not in features_to_drop
]

X_pruned = select_features(
    X_ext,
    PRUNED_FEATURE_NAMES
)

print("Full feature count:", len(EXTENDED_FEATURE_NAMES))
print("Pruned feature count:", len(PRUNED_FEATURE_NAMES))

Full feature count: 33
Pruned feature count: 23


In [26]:
catboost_pruned_result = evaluate_catboost(
    X_pruned,
    y,
    CV_SPLITS
)

Fold 0: 1.0319
Fold 1: 1.0310
Fold 2: 1.0265
Fold 3: 1.0260
Fold 4: 1.0317

OOF log loss: 1.0294
Fold mean:     1.0294
Fold std:      0.0026


In [27]:
print(
    "Full 33:",
    catboost_ext_result["log_loss"]
)

print(
    "Pruned 23:",
    catboost_pruned_result["log_loss"]
)

print(
    "Difference:",
    catboost_pruned_result["log_loss"]
    - catboost_ext_result["log_loss"]
)

Full 33: 1.0295256993094202
Pruned 23: 1.0294168418852543
Difference: -0.00010885742416588684


In [28]:
lr_pruned_result = evaluate_logistic_regression(
    X_pruned,
    y,
    CV_SPLITS
)

print(
    "LR + 23 features:",
    lr_pruned_result["log_loss"]
)

Fold 0: 1.0526
Fold 1: 1.0537
Fold 2: 1.0475
Fold 3: 1.0505
Fold 4: 1.0549

OOF log loss: 1.0519
Fold mean:     1.0519
Fold std:      0.0026
LR + 23 features: 1.0518577838976113


In [29]:
IMPORTANT_FEATURES = [
    "len_ratio_a_b",
    "log_len_a",
    "log_len_b",
    "similar_length",
    "length_a",
    "length_b",
    "refusals_a",
    "refusals_b",
    "repetition_density_a",
    "repetition_density_b",
]

In [30]:
individual_ablation = []

baseline_loss = catboost_pruned_result["log_loss"]

for feature in IMPORTANT_FEATURES:

    kept_features = [
        name
        for name in PRUNED_FEATURE_NAMES
        if name != feature
    ]

    X_without_feature = select_features(
        X_ext,
        kept_features
    )

    print()
    print("=" * 60)
    print(f"Removing feature: {feature}")
    print("=" * 60)

    result = evaluate_catboost(
        X_without_feature,
        y,
        CV_SPLITS
    )

    loss_without = result["log_loss"]

    individual_ablation.append({
        "removed_feature": feature,
        "log_loss_without": loss_without,
        "delta_vs_23": loss_without - baseline_loss,
    })


Removing feature: len_ratio_a_b
Fold 0: 1.0326
Fold 1: 1.0327
Fold 2: 1.0277
Fold 3: 1.0279
Fold 4: 1.0322

OOF log loss: 1.0306
Fold mean:     1.0306
Fold std:      0.0023

Removing feature: log_len_a
Fold 0: 1.0313
Fold 1: 1.0306
Fold 2: 1.0268
Fold 3: 1.0263
Fold 4: 1.0310

OOF log loss: 1.0292
Fold mean:     1.0292
Fold std:      0.0022

Removing feature: log_len_b
Fold 0: 1.0315
Fold 1: 1.0311
Fold 2: 1.0272
Fold 3: 1.0258
Fold 4: 1.0311

OOF log loss: 1.0293
Fold mean:     1.0293
Fold std:      0.0024

Removing feature: similar_length
Fold 0: 1.0318
Fold 1: 1.0313
Fold 2: 1.0268
Fold 3: 1.0255
Fold 4: 1.0307

OOF log loss: 1.0292
Fold mean:     1.0292
Fold std:      0.0025

Removing feature: length_a
Fold 0: 1.0312
Fold 1: 1.0304
Fold 2: 1.0264
Fold 3: 1.0260
Fold 4: 1.0308

OOF log loss: 1.0289
Fold mean:     1.0289
Fold std:      0.0023

Removing feature: length_b
Fold 0: 1.0311
Fold 1: 1.0310
Fold 2: 1.0264
Fold 3: 1.0266
Fold 4: 1.0311

OOF log loss: 1.0292
Fold mean:     1.

In [31]:
individual_ablation_df = (
    pd.DataFrame(individual_ablation)
    .sort_values("delta_vs_23", ascending=False)
)

individual_ablation_df

,removed_feature,log_loss_without,delta_vs_23
6,refusals_a,1.033577,0.004160
7,refusals_b,1.033572,0.004155
8,repetition_density_a,1.032646,0.003229
9,repetition_density_b,1.032576,0.003160
0,len_ratio_a_b,1.030600,0.001183
2,log_len_b,1.029326,-0.000090
5,length_b,1.029248,-0.000169
1,log_len_a,1.029214,-0.000203
3,similar_length,1.029208,-0.000209
4,length_a,1.028948,-0.000469


In [32]:
LENGTH_COMPONENTS = {
    "raw_lengths": [
        "length_a",
        "length_b",
    ],

    "log_lengths": [
        "log_len_a",
        "log_len_b",
    ],

    "length_ratio": [
        "len_ratio_a_b",
    ],

    "similar_length": [
        "similar_length",
    ],
}

In [33]:
length_ablation = []

baseline_loss = catboost_pruned_result["log_loss"]

for component_name, component_features in LENGTH_COMPONENTS.items():

    kept_features = [
        name
        for name in PRUNED_FEATURE_NAMES
        if name not in component_features
    ]

    X_without_component = select_features(
        X_ext,
        kept_features
    )

    print()
    print("=" * 60)
    print(f"Removing length component: {component_name}")
    print(f"Removed: {component_features}")
    print("=" * 60)

    result = evaluate_catboost(
        X_without_component,
        y,
        CV_SPLITS
    )

    loss_without = result["log_loss"]

    length_ablation.append({
        "removed_component": component_name,
        "log_loss_without": loss_without,
        "delta_vs_23": loss_without - baseline_loss,
        "n_removed": len(component_features),
    })


Removing length component: raw_lengths
Removed: ['length_a', 'length_b']
Fold 0: 1.0321
Fold 1: 1.0312
Fold 2: 1.0274
Fold 3: 1.0259
Fold 4: 1.0310

OOF log loss: 1.0295
Fold mean:     1.0295
Fold std:      0.0024

Removing length component: log_lengths
Removed: ['log_len_a', 'log_len_b']
Fold 0: 1.0312
Fold 1: 1.0309
Fold 2: 1.0268
Fold 3: 1.0257
Fold 4: 1.0306

OOF log loss: 1.0290
Fold mean:     1.0290
Fold std:      0.0023

Removing length component: length_ratio
Removed: ['len_ratio_a_b']
Fold 0: 1.0326
Fold 1: 1.0327
Fold 2: 1.0277
Fold 3: 1.0279
Fold 4: 1.0322

OOF log loss: 1.0306
Fold mean:     1.0306
Fold std:      0.0023

Removing length component: similar_length
Removed: ['similar_length']
Fold 0: 1.0318
Fold 1: 1.0313
Fold 2: 1.0268
Fold 3: 1.0255
Fold 4: 1.0307

OOF log loss: 1.0292
Fold mean:     1.0292
Fold std:      0.0025


In [34]:
length_ablation_df = (
    pd.DataFrame(length_ablation)
    .sort_values("delta_vs_23", ascending=False)
)

length_ablation_df

,removed_component,log_loss_without,delta_vs_23,n_removed
2,length_ratio,1.030600,0.001183,1
0,raw_lengths,1.029520,0.000103,2
3,similar_length,1.029208,-0.000209,1
1,log_lengths,1.029035,-0.000381,2


In [35]:
LENGTH_ALL = [
    "len_ratio_a_b",
    "log_len_a",
    "log_len_b",
    "similar_length",
    "length_a",
    "length_b",
]

LENGTH_VARIANTS = {
    "full_length": [
        "len_ratio_a_b",
        "log_len_a",
        "log_len_b",
        "similar_length",
        "length_a",
        "length_b",
    ],

    "ratio_raw": [
        "len_ratio_a_b",
        "length_a",
        "length_b",
    ],

    "ratio_log": [
        "len_ratio_a_b",
        "log_len_a",
        "log_len_b",
    ],

    "ratio_only": [
        "len_ratio_a_b",
    ],
}

In [36]:
NON_LENGTH_FEATURES = [
    name
    for name in PRUNED_FEATURE_NAMES
    if name not in LENGTH_ALL
]

print("Non-length features:", len(NON_LENGTH_FEATURES))

Non-length features: 17


In [37]:
length_variant_results = []

for variant_name, length_features in LENGTH_VARIANTS.items():

    feature_names = (
        NON_LENGTH_FEATURES
        + length_features
    )

    X_variant = select_features(
        X_ext,
        feature_names
    )

    print()
    print("=" * 60)
    print(f"Length variant: {variant_name}")
    print(f"Number of features: {len(feature_names)}")
    print(f"Length features: {length_features}")
    print("=" * 60)

    result = evaluate_catboost(
        X_variant,
        y,
        CV_SPLITS
    )

    length_variant_results.append({
        "variant": variant_name,
        "n_features": len(feature_names),
        "log_loss": result["log_loss"],
        "delta_vs_23": (
            result["log_loss"]
            - catboost_pruned_result["log_loss"]
        ),
    })


Length variant: full_length
Number of features: 23
Length features: ['len_ratio_a_b', 'log_len_a', 'log_len_b', 'similar_length', 'length_a', 'length_b']
Fold 0: 1.0321
Fold 1: 1.0307
Fold 2: 1.0265
Fold 3: 1.0269
Fold 4: 1.0312

OOF log loss: 1.0295
Fold mean:     1.0295
Fold std:      0.0023

Length variant: ratio_raw
Number of features: 20
Length features: ['len_ratio_a_b', 'length_a', 'length_b']
Fold 0: 1.0325
Fold 1: 1.0308
Fold 2: 1.0267
Fold 3: 1.0263
Fold 4: 1.0293

OOF log loss: 1.0291
Fold mean:     1.0291
Fold std:      0.0024

Length variant: ratio_log
Number of features: 20
Length features: ['len_ratio_a_b', 'log_len_a', 'log_len_b']
Fold 0: 1.0325
Fold 1: 1.0308
Fold 2: 1.0267
Fold 3: 1.0263
Fold 4: 1.0293

OOF log loss: 1.0291
Fold mean:     1.0291
Fold std:      0.0024

Length variant: ratio_only
Number of features: 18
Length features: ['len_ratio_a_b']
Fold 0: 1.0337
Fold 1: 1.0314
Fold 2: 1.0290
Fold 3: 1.0271
Fold 4: 1.0318

OOF log loss: 1.0306
Fold mean:     1.03

In [38]:
length_variants_df = (
    pd.DataFrame(length_variant_results)
    .sort_values("log_loss")
)

length_variants_df

,variant,n_features,log_loss,delta_vs_23
1,ratio_raw,20,1.029134,-0.000283
2,ratio_log,20,1.029134,-0.000283
0,full_length,23,1.029490,0.000073
3,ratio_only,18,1.030591,0.001174


In [39]:
FINAL_LENGTH_FEATURE_NAMES = (
    NON_LENGTH_FEATURES
    + [
        "len_ratio_a_b",
        "length_a",
        "length_b",
    ]
)

X_20 = select_features(
    X_ext,
    FINAL_LENGTH_FEATURE_NAMES
)

print("Number of features:", len(FINAL_LENGTH_FEATURE_NAMES))

Number of features: 20


In [40]:
BASELINE_20_LOSS = (
    length_variants_df
    .loc[
        length_variants_df["variant"] == "ratio_raw",
        "log_loss"
    ]
    .iloc[0]
)

print("20-feature baseline:", BASELINE_20_LOSS)

20-feature baseline: 1.029133925300099


In [41]:
PARAGRAPH_PAIRS = {
    "num_paragraphs": [
        "num_paragraphs_a",
        "num_paragraphs_b",
    ],

    "mean_paragraph_length": [
        "mean_paragraph_length_a",
        "mean_paragraph_length_b",
    ],

    "sd_paragraph_length": [
        "sd_paragraph_length_a",
        "sd_paragraph_length_b",
    ],

    "avg_sent_per_paragraph": [
        "avg_sent_per_paragraph_a",
        "avg_sent_per_paragraph_b",
    ],

    "sd_sent_per_paragraph": [
        "sd_sent_per_paragraph_a",
        "sd_sent_per_paragraph_b",
    ],
}

In [43]:
paragraph_pair_ablation = []

for pair_name, pair_features in PARAGRAPH_PAIRS.items():

    kept_features = [
        name
        for name in FINAL_LENGTH_FEATURE_NAMES
        if name not in pair_features
    ]

    X_without_pair = select_features(
        X_ext,
        kept_features
    )

    print()
    print("=" * 60)
    print(f"Removing paragraph pair: {pair_name}")
    print(f"Removed: {pair_features}")
    print(f"Number of features: {len(kept_features)}")
    print("=" * 60)

    result = evaluate_catboost(
        X_without_pair,
        y,
        CV_SPLITS
    )

    loss_without = result["log_loss"]

    paragraph_pair_ablation.append({
        "removed_pair": pair_name,
        "log_loss_without": loss_without,
        "delta_vs_20": loss_without - BASELINE_20_LOSS,
    })


Removing paragraph pair: num_paragraphs
Removed: ['num_paragraphs_a', 'num_paragraphs_b']
Number of features: 18
Fold 0: 1.0318
Fold 1: 1.0315
Fold 2: 1.0267
Fold 3: 1.0266
Fold 4: 1.0311

OOF log loss: 1.0295
Fold mean:     1.0295
Fold std:      0.0024

Removing paragraph pair: mean_paragraph_length
Removed: ['mean_paragraph_length_a', 'mean_paragraph_length_b']
Number of features: 18
Fold 0: 1.0317
Fold 1: 1.0312
Fold 2: 1.0266
Fold 3: 1.0253
Fold 4: 1.0305

OOF log loss: 1.0290
Fold mean:     1.0290
Fold std:      0.0026

Removing paragraph pair: sd_paragraph_length
Removed: ['sd_paragraph_length_a', 'sd_paragraph_length_b']
Number of features: 18
Fold 0: 1.0322
Fold 1: 1.0311
Fold 2: 1.0274
Fold 3: 1.0248
Fold 4: 1.0305

OOF log loss: 1.0292
Fold mean:     1.0292
Fold std:      0.0027

Removing paragraph pair: avg_sent_per_paragraph
Removed: ['avg_sent_per_paragraph_a', 'avg_sent_per_paragraph_b']
Number of features: 18
Fold 0: 1.0323
Fold 1: 1.0297
Fold 2: 1.0273
Fold 3: 1.0262
F

In [44]:
paragraph_pair_ablation_df = (
    pd.DataFrame(paragraph_pair_ablation)
    .sort_values(
        "delta_vs_20",
        ascending=False
    )
)

paragraph_pair_ablation_df

,removed_pair,log_loss_without,delta_vs_20
0,num_paragraphs,1.029535,0.000401
3,avg_sent_per_paragraph,1.029253,0.000119
2,sd_paragraph_length,1.029202,0.000068
1,mean_paragraph_length,1.029033,-0.000100
4,sd_sent_per_paragraph,1.028735,-0.000399


In [46]:
PARAGRAPH_DROP_1 = [
    "sd_sent_per_paragraph_a",
    "sd_sent_per_paragraph_b",
]

FEATURE_NAMES_18 = [
    name
    for name in FINAL_LENGTH_FEATURE_NAMES
    if name not in PARAGRAPH_DROP_1
]

X_18 = select_features(
    X_ext,
    FEATURE_NAMES_18
)

print("Number of features:", len(FEATURE_NAMES_18))

catboost_18_result = evaluate_catboost(
    X_18,
    y,
    CV_SPLITS
)

print("20 features:", BASELINE_20_LOSS)
print("18 features:", catboost_18_result["log_loss"])
print(
    "Difference:",
    catboost_18_result["log_loss"] - BASELINE_20_LOSS
)

Number of features: 18
Fold 0: 1.0315
Fold 1: 1.0300
Fold 2: 1.0262
Fold 3: 1.0255
Fold 4: 1.0304

OOF log loss: 1.0287
Fold mean:     1.0287
Fold std:      0.0024
20 features: 1.029133925300099
18 features: 1.028734503856412
Difference: -0.000399421443687098


In [47]:
BASELINE_18_LOSS = catboost_18_result["log_loss"]

print("18-feature baseline:", BASELINE_18_LOSS)
print("Number of features:", len(FEATURE_NAMES_18))

18-feature baseline: 1.028734503856412
Number of features: 18


In [48]:
PARAGRAPH_PAIRS_18 = {
    "num_paragraphs": [
        "num_paragraphs_a",
        "num_paragraphs_b",
    ],

    "mean_paragraph_length": [
        "mean_paragraph_length_a",
        "mean_paragraph_length_b",
    ],

    "sd_paragraph_length": [
        "sd_paragraph_length_a",
        "sd_paragraph_length_b",
    ],

    "avg_sent_per_paragraph": [
        "avg_sent_per_paragraph_a",
        "avg_sent_per_paragraph_b",
    ],
}

In [49]:
paragraph_ablation_18 = []

for pair_name, pair_features in PARAGRAPH_PAIRS_18.items():

    kept_features = [
        name
        for name in FEATURE_NAMES_18
        if name not in pair_features
    ]

    X_without_pair = select_features(
        X_ext,
        kept_features
    )

    print()
    print("=" * 60)
    print(f"Removing paragraph pair: {pair_name}")
    print(f"Removed: {pair_features}")
    print(f"Features left: {len(kept_features)}")
    print("=" * 60)

    result = evaluate_catboost(
        X_without_pair,
        y,
        CV_SPLITS
    )

    loss_without = result["log_loss"]

    paragraph_ablation_18.append({
        "removed_pair": pair_name,
        "log_loss_without": loss_without,
        "delta_vs_18": loss_without - BASELINE_18_LOSS,
    })


Removing paragraph pair: num_paragraphs
Removed: ['num_paragraphs_a', 'num_paragraphs_b']
Features left: 16
Fold 0: 1.0318
Fold 1: 1.0319
Fold 2: 1.0264
Fold 3: 1.0259
Fold 4: 1.0304

OOF log loss: 1.0293
Fold mean:     1.0293
Fold std:      0.0026

Removing paragraph pair: mean_paragraph_length
Removed: ['mean_paragraph_length_a', 'mean_paragraph_length_b']
Features left: 16
Fold 0: 1.0314
Fold 1: 1.0322
Fold 2: 1.0262
Fold 3: 1.0248
Fold 4: 1.0306

OOF log loss: 1.0290
Fold mean:     1.0290
Fold std:      0.0030

Removing paragraph pair: sd_paragraph_length
Removed: ['sd_paragraph_length_a', 'sd_paragraph_length_b']
Features left: 16
Fold 0: 1.0314
Fold 1: 1.0314
Fold 2: 1.0262
Fold 3: 1.0251
Fold 4: 1.0308

OOF log loss: 1.0290
Fold mean:     1.0290
Fold std:      0.0027

Removing paragraph pair: avg_sent_per_paragraph
Removed: ['avg_sent_per_paragraph_a', 'avg_sent_per_paragraph_b']
Features left: 16
Fold 0: 1.0317
Fold 1: 1.0314
Fold 2: 1.0268
Fold 3: 1.0257
Fold 4: 1.0307

OOF l

In [50]:
paragraph_ablation_18_df = (
    pd.DataFrame(paragraph_ablation_18)
    .sort_values(
        "delta_vs_18",
        ascending=False
    )
)

paragraph_ablation_18_df

,removed_pair,log_loss_without,delta_vs_18
0,num_paragraphs,1.029273,0.000539
3,avg_sent_per_paragraph,1.029263,0.000528
1,mean_paragraph_length,1.029024,0.000290
2,sd_paragraph_length,1.028985,0.000251


In [51]:
PARAGRAPH_ALL = [
    "num_paragraphs_a",
    "num_paragraphs_b",
    "mean_paragraph_length_a",
    "mean_paragraph_length_b",
    "sd_paragraph_length_a",
    "sd_paragraph_length_b",
    "avg_sent_per_paragraph_a",
    "avg_sent_per_paragraph_b",
]

PARAGRAPH_VARIANTS = {
    # текущая версия
    "all_8": PARAGRAPH_ALL,

    # убираем только самую слабую пару
    "without_sd_length": [
        "num_paragraphs_a",
        "num_paragraphs_b",
        "mean_paragraph_length_a",
        "mean_paragraph_length_b",
        "avg_sent_per_paragraph_a",
        "avg_sent_per_paragraph_b",
    ],

    # оставляем только две наиболее полезные пары
    "core_4": [
        "num_paragraphs_a",
        "num_paragraphs_b",
        "avg_sent_per_paragraph_a",
        "avg_sent_per_paragraph_b",
    ],

    # совсем минимальный вариант
    "num_paragraphs_only": [
        "num_paragraphs_a",
        "num_paragraphs_b",
    ],
}

In [52]:
NON_PARAGRAPH_FEATURES = [
    name
    for name in FEATURE_NAMES_18
    if name not in PARAGRAPH_ALL
]

print("Non-paragraph features:", len(NON_PARAGRAPH_FEATURES))

Non-paragraph features: 10


In [55]:
paragraph_variant_results = []

for variant_name, paragraph_features in PARAGRAPH_VARIANTS.items():

    paragraph_features = set(paragraph_features)

    # IMPORTANT:
    # preserve exactly the same feature order as FEATURE_NAMES_18
    feature_names = [
        name
        for name in FEATURE_NAMES_18
        if (
            name not in PARAGRAPH_ALL
            or name in paragraph_features
        )
    ]

    X_variant = select_features(
        X_ext,
        feature_names
    )

    print()
    print("=" * 60)
    print(f"Paragraph variant: {variant_name}")
    print(f"Total features: {len(feature_names)}")
    print(f"Paragraph features: {len(paragraph_features)}")
    print("=" * 60)

    result = evaluate_catboost(
        X_variant,
        y,
        CV_SPLITS
    )

    paragraph_variant_results.append({
        "variant": variant_name,
        "n_total_features": len(feature_names),
        "n_paragraph_features": len(paragraph_features),
        "log_loss": result["log_loss"],
        "delta_vs_18": (
            result["log_loss"]
            - BASELINE_18_LOSS
        ),
    })


paragraph_variants_df = (
    pd.DataFrame(paragraph_variant_results)
    .sort_values("log_loss")
)

paragraph_variants_df


Paragraph variant: all_8
Total features: 18
Paragraph features: 8
Fold 0: 1.0315
Fold 1: 1.0300
Fold 2: 1.0262
Fold 3: 1.0255
Fold 4: 1.0304

OOF log loss: 1.0287
Fold mean:     1.0287
Fold std:      0.0024

Paragraph variant: without_sd_length
Total features: 16
Paragraph features: 6
Fold 0: 1.0314
Fold 1: 1.0314
Fold 2: 1.0262
Fold 3: 1.0251
Fold 4: 1.0308

OOF log loss: 1.0290
Fold mean:     1.0290
Fold std:      0.0027

Paragraph variant: core_4
Total features: 14
Paragraph features: 4
Fold 0: 1.0320
Fold 1: 1.0309
Fold 2: 1.0267
Fold 3: 1.0259
Fold 4: 1.0310

OOF log loss: 1.0293
Fold mean:     1.0293
Fold std:      0.0025

Paragraph variant: num_paragraphs_only
Total features: 12
Paragraph features: 2
Fold 0: 1.0314
Fold 1: 1.0318
Fold 2: 1.0262
Fold 3: 1.0262
Fold 4: 1.0315

OOF log loss: 1.0294
Fold mean:     1.0294
Fold std:      0.0026


,variant,n_total_features,n_paragraph_features,log_loss,delta_vs_18
0,all_8,18,8,1.028735,0.000000
1,without_sd_length,16,6,1.028985,0.000251
2,core_4,14,4,1.029302,0.000568
3,num_paragraphs_only,12,2,1.029440,0.000706


In [58]:
SMALL_GROUPS_18 = {
    "prompt_length": [
        "prompt_length",
    ],

    "punctuation": [
        "punctuation_density_a",
        "punctuation_density_b",
    ],
}

In [59]:
small_group_ablation = []

BASELINE_FINAL_18 = catboost_18_result["log_loss"]

for group_name, group_features in SMALL_GROUPS_18.items():

    kept_features = [
        name
        for name in FEATURE_NAMES_18
        if name not in group_features
    ]

    X_without_group = select_features(
        X_ext,
        kept_features
    )

    print()
    print("=" * 60)
    print(f"Removing group: {group_name}")
    print(f"Removed: {group_features}")
    print(f"Features left: {len(kept_features)}")
    print("=" * 60)

    result = evaluate_catboost(
        X_without_group,
        y,
        CV_SPLITS
    )

    loss_without = result["log_loss"]

    small_group_ablation.append({
        "removed_group": group_name,
        "n_features": len(kept_features),
        "log_loss_without": loss_without,
        "delta_vs_18": loss_without - BASELINE_FINAL_18,
    })


Removing group: prompt_length
Removed: ['prompt_length']
Features left: 17
Fold 0: 1.0340
Fold 1: 1.0338
Fold 2: 1.0282
Fold 3: 1.0272
Fold 4: 1.0328

OOF log loss: 1.0312
Fold mean:     1.0312
Fold std:      0.0029

Removing group: punctuation
Removed: ['punctuation_density_a', 'punctuation_density_b']
Features left: 16
Fold 0: 1.0331
Fold 1: 1.0324
Fold 2: 1.0285
Fold 3: 1.0271
Fold 4: 1.0325

OOF log loss: 1.0307
Fold mean:     1.0307
Fold std:      0.0024


In [60]:
small_group_ablation_df = (
    pd.DataFrame(small_group_ablation)
    .sort_values("delta_vs_18", ascending=False)
)

small_group_ablation_df

,removed_group,n_features,log_loss_without,delta_vs_18
0,prompt_length,17,1.031197,0.002463
1,punctuation,16,1.030723,0.001988


In [61]:
drop_both = [
    "prompt_length",
    "punctuation_density_a",
    "punctuation_density_b",
]

FEATURE_NAMES_15 = [
    name
    for name in FEATURE_NAMES_18
    if name not in drop_both
]

X_15 = select_features(
    X_ext,
    FEATURE_NAMES_15
)

catboost_15_result = evaluate_catboost(
    X_15,
    y,
    CV_SPLITS
)

print("18 features:", BASELINE_FINAL_18)
print("15 features:", catboost_15_result["log_loss"])
print(
    "Difference:",
    catboost_15_result["log_loss"] - BASELINE_FINAL_18
)

Fold 0: 1.0353
Fold 1: 1.0364
Fold 2: 1.0303
Fold 3: 1.0290
Fold 4: 1.0360

OOF log loss: 1.0334
Fold mean:     1.0334
Fold std:      0.0031
18 features: 1.028734503856412
15 features: 1.0333932028491133
Difference: 0.004658698992701327


In [62]:
X_18 = select_features(
    X_ext,
    FEATURE_NAMES_18
)

lr_18_result = evaluate_logistic_regression(
    X_18,
    y,
    CV_SPLITS
)

print(
    "LR + 18 features:",
    lr_18_result["log_loss"]
)

Fold 0: 1.0574
Fold 1: 1.0592
Fold 2: 1.0553
Fold 3: 1.0565
Fold 4: 1.0592

OOF log loss: 1.0575
Fold mean:     1.0575
Fold std:      0.0015
LR + 18 features: 1.0575189637590965


In [63]:
final_comparison = pd.DataFrame({
    "model": [
        "Logistic Regression + 9 features",
        "CatBoost + 9 features",
        "Logistic Regression + 18 features",
        "CatBoost + 18 features",
    ],

    "log_loss": [
        lr_9_result["log_loss"],
        catboost_9_result["log_loss"],
        lr_18_result["log_loss"],
        catboost_18_result["log_loss"],
    ],
})

final_comparison["improvement_vs_lr9"] = (
    final_comparison["log_loss"]
    - lr_9_result["log_loss"]
)

final_comparison.sort_values("log_loss")

,model,log_loss,improvement_vs_lr9
3,CatBoost + 18 features,1.028735,-0.033959
1,CatBoost + 9 features,1.041892,-0.020801
2,Logistic Regression + 18 features,1.057519,-0.005174
0,Logistic Regression + 9 features,1.062693,0.000000


In [66]:
confirmation_skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

CONFIRMATION_SPLITS = list(
    confirmation_skf.split(np.zeros(len(y)), y)
)

In [67]:
confirm_lr_9 = evaluate_logistic_regression(
    X9,
    y,
    CONFIRMATION_SPLITS
)

confirm_catboost_9 = evaluate_catboost(
    X9,
    y,
    CONFIRMATION_SPLITS
)

confirm_lr_18 = evaluate_logistic_regression(
    X_18,
    y,
    CONFIRMATION_SPLITS
)

confirm_catboost_18 = evaluate_catboost(
    X_18,
    y,
    CONFIRMATION_SPLITS
)

Fold 0: 1.0638
Fold 1: 1.0657
Fold 2: 1.0620
Fold 3: 1.0620
Fold 4: 1.0602

OOF log loss: 1.0628
Fold mean:     1.0628
Fold std:      0.0019
Fold 0: 1.0420
Fold 1: 1.0456
Fold 2: 1.0405
Fold 3: 1.0406
Fold 4: 1.0396

OOF log loss: 1.0417
Fold mean:     1.0416
Fold std:      0.0021
Fold 0: 1.0604
Fold 1: 1.0610
Fold 2: 1.0553
Fold 3: 1.0558
Fold 4: 1.0540

OOF log loss: 1.0573
Fold mean:     1.0573
Fold std:      0.0029
Fold 0: 1.0298
Fold 1: 1.0346
Fold 2: 1.0268
Fold 3: 1.0274
Fold 4: 1.0266

OOF log loss: 1.0290
Fold mean:     1.0290
Fold std:      0.0030


In [68]:
confirmation_comparison = pd.DataFrame({
    "model": [
        "Logistic Regression + 9 features",
        "CatBoost + 9 features",
        "Logistic Regression + 18 features",
        "CatBoost + 18 features",
    ],

    "original_cv": [
        lr_9_result["log_loss"],
        catboost_9_result["log_loss"],
        lr_18_result["log_loss"],
        catboost_18_result["log_loss"],
    ],

    "confirmation_cv": [
        confirm_lr_9["log_loss"],
        confirm_catboost_9["log_loss"],
        confirm_lr_18["log_loss"],
        confirm_catboost_18["log_loss"],
    ],
})

confirmation_comparison["difference"] = (
    confirmation_comparison["confirmation_cv"]
    - confirmation_comparison["original_cv"]
)

confirmation_comparison

,model,original_cv,confirmation_cv,difference
0,Logistic Regression + 9 features,1.062693,1.062750,0.000057
1,CatBoost + 9 features,1.041892,1.041650,-0.000242
2,Logistic Regression + 18 features,1.057519,1.057304,-0.000215
3,CatBoost + 18 features,1.028735,1.029040,0.000306


In [69]:
NUMBER_RE = re.compile(
    r"(?<!\w)[+-]?\d+(?:[.,]\d+)?%?(?!\w)"
)


def numeric_density(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0

    n_words = len(get_words(text))

    if n_words == 0:
        return 0.0

    n_numbers = len(NUMBER_RE.findall(text))

    return n_numbers / n_words

In [70]:
num_density_a = tr["response_a_txt"].apply(
    numeric_density
).to_numpy(float)

num_density_b = tr["response_b_txt"].apply(
    numeric_density
).to_numpy(float)

numeric_density_diff = (
    num_density_a - num_density_b
)

In [71]:
EXPLANATION_RE = re.compile(
    r"\b("
    r"because|"
    r"therefore|"
    r"thus|"
    r"hence|"
    r"due to|"
    r"as a result|"
    r"this means|"
    r"which means|"
    r"the reason|"
    r"in other words|"
    r"for example|"
    r"for instance"
    r")\b",
    re.IGNORECASE,
)


def explanation_marker_density(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0

    n_words = len(get_words(text))

    if n_words == 0:
        return 0.0

    n_markers = len(
        EXPLANATION_RE.findall(text)
    )

    return n_markers / n_words

In [72]:
expl_density_a = tr["response_a_txt"].apply(
    explanation_marker_density
).to_numpy(float)

expl_density_b = tr["response_b_txt"].apply(
    explanation_marker_density
).to_numpy(float)

explanation_density_diff = (
    expl_density_a - expl_density_b
)

In [73]:
num_count_a = tr["response_a_txt"].apply(
    lambda x: len(NUMBER_RE.findall(x))
)

num_count_b = tr["response_b_txt"].apply(
    lambda x: len(NUMBER_RE.findall(x))
)

only_a_numbers = (
    (num_count_a > 0)
    & (num_count_b == 0)
)

only_b_numbers = (
    (num_count_b > 0)
    & (num_count_a == 0)
)

In [74]:
def show_outcomes(mask, title):
    subset = tr.loc[mask]

    shares = (
        subset[
            [
                "winner_model_a",
                "winner_model_b",
                "winner_tie",
            ]
        ]
        .mean()
    )

    print(title)
    print("n =", len(subset))
    print(f"A wins: {shares['winner_model_a']:.1%}")
    print(f"B wins: {shares['winner_model_b']:.1%}")
    print(f"Tie:    {shares['winner_tie']:.1%}")
    print()


show_outcomes(
    only_a_numbers,
    "Only A contains numbers"
)

show_outcomes(
    only_b_numbers,
    "Only B contains numbers"
)

Only A contains numbers
n = 6752
A wins: 48.8%
B wins: 25.3%
Tie:    25.9%

Only B contains numbers
n = 6685
A wins: 26.0%
B wins: 48.2%
Tie:    25.8%



In [76]:
NEW_FEATURE_NAMES = [
    "numeric_density_diff",
    "explanation_density_diff",
]

X_20_new = np.column_stack([
    X_18,
    numeric_density_diff,
    explanation_density_diff,
])

FEATURE_NAMES_20 = (
    FEATURE_NAMES_18
    + NEW_FEATURE_NAMES
)

print(X_20_new.shape)
print(len(FEATURE_NAMES_20))

(57477, 20)
20


In [77]:
catboost_20_result = evaluate_catboost(
    X_20_new,
    y,
    CV_SPLITS
)

print(
    "CatBoost + 18:",
    catboost_18_result["log_loss"]
)

print(
    "CatBoost + 20:",
    catboost_20_result["log_loss"]
)

print(
    "Effect:",
    catboost_20_result["log_loss"]
    - catboost_18_result["log_loss"]
)

Fold 0: 1.0318
Fold 1: 1.0322
Fold 2: 1.0257
Fold 3: 1.0263
Fold 4: 1.0301

OOF log loss: 1.0292
Fold mean:     1.0292
Fold std:      0.0027
CatBoost + 18: 1.028734503856412
CatBoost + 20: 1.029231313645152
Effect: 0.0004968097887401157


In [78]:
lr_20_result = evaluate_logistic_regression(
    X_20_new,
    y,
    CV_SPLITS
)

Fold 0: 1.0567
Fold 1: 1.0585
Fold 2: 1.0543
Fold 3: 1.0557
Fold 4: 1.0588

OOF log loss: 1.0568
Fold mean:     1.0568
Fold std:      0.0017


In [79]:
NEW_FEATURE_VARIANTS = {
    "baseline_18": [],
    "numeric_only": [
        "numeric_density_diff"
    ],
    "explanation_only": [
        "explanation_density_diff"
    ],
    "both_20": [
        "numeric_density_diff",
        "explanation_density_diff",
    ],
}

In [80]:
new_feature_results = []

variants = {
    "baseline_18": X_18,

    "numeric_only": np.column_stack([
        X_18,
        numeric_density_diff,
    ]),

    "explanation_only": np.column_stack([
        X_18,
        explanation_density_diff,
    ]),

    "both_20": X_20_new,
}


for name, X_variant in variants.items():

    result = evaluate_catboost(
        X_variant,
        y,
        CV_SPLITS
    )

    new_feature_results.append({
        "variant": name,
        "n_features": X_variant.shape[1],
        "log_loss": result["log_loss"],
        "delta_vs_18": (
            result["log_loss"]
            - catboost_18_result["log_loss"]
        ),
    })


new_feature_df = (
    pd.DataFrame(new_feature_results)
    .sort_values("log_loss")
)

new_feature_df

Fold 0: 1.0315
Fold 1: 1.0300
Fold 2: 1.0262
Fold 3: 1.0255
Fold 4: 1.0304

OOF log loss: 1.0287
Fold mean:     1.0287
Fold std:      0.0024
Fold 0: 1.0322
Fold 1: 1.0317
Fold 2: 1.0257
Fold 3: 1.0247
Fold 4: 1.0292

OOF log loss: 1.0287
Fold mean:     1.0287
Fold std:      0.0030
Fold 0: 1.0320
Fold 1: 1.0311
Fold 2: 1.0263
Fold 3: 1.0260
Fold 4: 1.0302

OOF log loss: 1.0291
Fold mean:     1.0291
Fold std:      0.0025
Fold 0: 1.0318
Fold 1: 1.0322
Fold 2: 1.0257
Fold 3: 1.0263
Fold 4: 1.0301

OOF log loss: 1.0292
Fold mean:     1.0292
Fold std:      0.0027


,variant,n_features,log_loss,delta_vs_18
1,numeric_only,19,1.028697,-0.000038
0,baseline_18,18,1.028735,0.000000
2,explanation_only,19,1.029110,0.000376
3,both_20,20,1.029231,0.000497


In [81]:
has_numbers_a = (
    num_count_a > 0
).astype(float).to_numpy()

has_numbers_b = (
    num_count_b > 0
).astype(float).to_numpy()

In [82]:
X_20_numbers = np.column_stack([
    X_18,
    has_numbers_a,
    has_numbers_b,
])

FEATURE_NAMES_20_NUMBERS = (
    FEATURE_NAMES_18
    + [
        "has_numbers_a",
        "has_numbers_b",
    ]
)

print(X_20_numbers.shape)

(57477, 20)


In [83]:
catboost_20_numbers_result = evaluate_catboost(
    X_20_numbers,
    y,
    CV_SPLITS
)

print(
    "18 features:",
    catboost_18_result["log_loss"]
)

print(
    "20 features (number presence):",
    catboost_20_numbers_result["log_loss"]
)

print(
    "Difference:",
    catboost_20_numbers_result["log_loss"]
    - catboost_18_result["log_loss"]
)

Fold 0: 1.0310
Fold 1: 1.0307
Fold 2: 1.0256
Fold 3: 1.0247
Fold 4: 1.0297

OOF log loss: 1.0283
Fold mean:     1.0283
Fold std:      0.0027
18 features: 1.028734503856412
20 features (number presence): 1.0283254763620793
Difference: -0.00040902749433269214


In [84]:
NUMBER_VARIANTS = {
    "baseline_18": X_18,

    "density_diff": np.column_stack([
        X_18,
        numeric_density_diff,
    ]),

    "presence_ab": np.column_stack([
        X_18,
        has_numbers_a,
        has_numbers_b,
    ]),

    "presence_plus_density": np.column_stack([
        X_18,
        has_numbers_a,
        has_numbers_b,
        numeric_density_diff,
    ]),
}

In [85]:
number_results = []

for name, X_variant in NUMBER_VARIANTS.items():

    result = evaluate_catboost(
        X_variant,
        y,
        CV_SPLITS
    )

    number_results.append({
        "variant": name,
        "n_features": X_variant.shape[1],
        "log_loss": result["log_loss"],
        "delta_vs_18": (
            result["log_loss"]
            - catboost_18_result["log_loss"]
        ),
    })


number_results_df = (
    pd.DataFrame(number_results)
    .sort_values("log_loss")
)

number_results_df

Fold 0: 1.0315
Fold 1: 1.0300
Fold 2: 1.0262
Fold 3: 1.0255
Fold 4: 1.0304

OOF log loss: 1.0287
Fold mean:     1.0287
Fold std:      0.0024
Fold 0: 1.0322
Fold 1: 1.0317
Fold 2: 1.0257
Fold 3: 1.0247
Fold 4: 1.0292

OOF log loss: 1.0287
Fold mean:     1.0287
Fold std:      0.0030
Fold 0: 1.0310
Fold 1: 1.0307
Fold 2: 1.0256
Fold 3: 1.0247
Fold 4: 1.0297

OOF log loss: 1.0283
Fold mean:     1.0283
Fold std:      0.0027
Fold 0: 1.0316
Fold 1: 1.0307
Fold 2: 1.0259
Fold 3: 1.0249
Fold 4: 1.0300

OOF log loss: 1.0286
Fold mean:     1.0286
Fold std:      0.0027


,variant,n_features,log_loss,delta_vs_18
2,presence_ab,20,1.028325,-0.000409
3,presence_plus_density,21,1.028620,-0.000114
1,density_diff,19,1.028697,-0.000038
0,baseline_18,18,1.028735,0.000000


In [86]:
FEATURE_NAMES_20_FINAL = (
    FEATURE_NAMES_18
    + [
        "has_numbers_a",
        "has_numbers_b",
    ]
)

X_20_final = np.column_stack([
    X_18,
    has_numbers_a,
    has_numbers_b,
])

print(X_20_final.shape)
print(FEATURE_NAMES_20_FINAL)

(57477, 20)
['refusals_a', 'refusals_b', 'prompt_length', 'num_paragraphs_a', 'mean_paragraph_length_a', 'sd_paragraph_length_a', 'avg_sent_per_paragraph_a', 'repetition_density_a', 'punctuation_density_a', 'num_paragraphs_b', 'mean_paragraph_length_b', 'sd_paragraph_length_b', 'avg_sent_per_paragraph_b', 'repetition_density_b', 'punctuation_density_b', 'len_ratio_a_b', 'length_a', 'length_b', 'has_numbers_a', 'has_numbers_b']


In [87]:
lr_20_final_result = evaluate_logistic_regression(
    X_20_final,
    y,
    CV_SPLITS
)

print(
    "LR + 20 features:",
    lr_20_final_result["log_loss"]
)

Fold 0: 1.0566
Fold 1: 1.0583
Fold 2: 1.0546
Fold 3: 1.0556
Fold 4: 1.0584

OOF log loss: 1.0567
Fold mean:     1.0567
Fold std:      0.0015
LR + 20 features: 1.056715331684141


In [88]:
final_skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=314159
)

FINAL_SPLITS = list(
    final_skf.split(np.zeros(len(y)), y)
)

In [89]:
final_lr_9 = evaluate_logistic_regression(
    X9,
    y,
    FINAL_SPLITS
)

final_catboost_9 = evaluate_catboost(
    X9,
    y,
    FINAL_SPLITS
)

final_lr_20 = evaluate_logistic_regression(
    X_20_final,
    y,
    FINAL_SPLITS
)

final_catboost_20 = evaluate_catboost(
    X_20_final,
    y,
    FINAL_SPLITS
)

Fold 0: 1.0622
Fold 1: 1.0610
Fold 2: 1.0643
Fold 3: 1.0632
Fold 4: 1.0631

OOF log loss: 1.0628
Fold mean:     1.0628
Fold std:      0.0011
Fold 0: 1.0437
Fold 1: 1.0409
Fold 2: 1.0411
Fold 3: 1.0429
Fold 4: 1.0435

OOF log loss: 1.0424
Fold mean:     1.0424
Fold std:      0.0012
Fold 0: 1.0575
Fold 1: 1.0547
Fold 2: 1.0580
Fold 3: 1.0563
Fold 4: 1.0553

OOF log loss: 1.0564
Fold mean:     1.0564
Fold std:      0.0013
Fold 0: 1.0288
Fold 1: 1.0267
Fold 2: 1.0273
Fold 3: 1.0288
Fold 4: 1.0323

OOF log loss: 1.0288
Fold mean:     1.0288
Fold std:      0.0019


In [90]:
final_robustness = pd.DataFrame({
    "model": [
        "Logistic Regression + 9 features",
        "CatBoost + 9 features",
        "Logistic Regression + 20 features",
        "CatBoost + 20 features",
    ],

    "selection_cv": [
        lr_9_result["log_loss"],
        catboost_9_result["log_loss"],
        lr_20_final_result["log_loss"],
        catboost_20_numbers_result["log_loss"],
    ],

    "final_cv": [
        final_lr_9["log_loss"],
        final_catboost_9["log_loss"],
        final_lr_20["log_loss"],
        final_catboost_20["log_loss"],
    ],
})

final_robustness["difference"] = (
    final_robustness["final_cv"]
    - final_robustness["selection_cv"]
)

final_robustness

,model,selection_cv,final_cv,difference
0,Logistic Regression + 9 features,1.062693,1.062759,0.000066
1,CatBoost + 9 features,1.041892,1.042413,0.000521
2,Logistic Regression + 20 features,1.056715,1.056391,-0.000325
3,CatBoost + 20 features,1.028325,1.028788,0.000463
